In [1]:
import os
import sys
import django
PATH_PROJECT = fr"D:\PROGRAMMING\Projects\siakad\siakad_django"
sys.path.append(PATH_PROJECT)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'siakad_django.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()
from django.db import transaction

from django.db.models import Prefetch, Q

from django.conf import settings

In [2]:
from academic.models import (
    MataKuliah, Kelas, PeriodeKRS, PenawaranKRS,
    KRS, Mahasiswa, DetailKRS
)

serializer

In [51]:

from academic.models import (
    MataKuliah, Kelas
)

from rest_framework import serializers

class KelasSerializer(serializers.ModelSerializer):
    class Meta:
        model = Kelas
        fields = [
            "id",
            "nama",
            # "kode",
            # "waktu",
            # "ruangan",
        ]
        

class MataKuliahAjukanBanyakSerializer(serializers.ModelSerializer):
    daftar_kelas = KelasSerializer(many=True, read_only=True)
    class Meta:
        model = MataKuliah
        fields =  [
            "id", "nama", "sks", "kode", "semester", "jenis",
            "daftar_kelas", 
        ]
        

class MahasiswaAjukanBanyakSerializerSerializer(serializers.ModelSerializer):
    class Meta:
        model = Mahasiswa
        fields = [
            "id",
            "nama",
            "nim", 
            "no_hp",
            "jenis_kelamin"
        ]


In [ ]:
from academic.krs.services.ajukanBanyak import AjukanBanyakService

query

In [4]:
periode_krs = PeriodeKRS.objects.get(status="aktif")
periode_krs

<PeriodeKRS: 1 - 2026/2027 - ganjil>

In [7]:
kelas_qs = (
    Kelas.objects
    .filter(
        penawarankrs__periode_krs__id=periode_krs.id
    )
)
kelas_qs

<QuerySet [<Kelas: 125 - 31MKP63AS19-PPW - PPW>, <Kelas: 126 - 11MKI63PA01-PPW - PPW>, <Kelas: 127 - 11MKI63AB04-PPW - PPW>, <Kelas: 128 - 11MKP63AK06-PPW - PPW>, <Kelas: 129 - 21MKP63AO16-PPW - PPW>, <Kelas: 130 - 21MKP63KB15-PPW - PPW>, <Kelas: 131 - 21MKP63BD14-PPW - PPW>, <Kelas: 132 - 41MKP63BC71-PPW - PPW>, <Kelas: 133 - 41MKP63CV57-PPW - PPW>, <Kelas: 134 - 41MKP63CC55-PPW - PPW>, <Kelas: 135 - 41MKP63SE65-PPW - PPW>, <Kelas: 136 - 41MKP63DF56-PPW - PPW>, <Kelas: 137 - 41MKP63BI62-PPW - PPW>, <Kelas: 138 - 41MKP63EC67-PPW - PPW>, <Kelas: 139 - 41MKP63EH61-PPW - PPW>, <Kelas: 140 - 41MKP63ER63-PPW - PPW>, <Kelas: 141 - 31MKP63IM03-PPW - PPW>, <Kelas: 142 - 21MKP63JK09-PPW - PPW>, <Kelas: 143 - 11MKP63KL01-PPW - PPW>, <Kelas: 144 - 41MKP63KP7-PPW - PPW>, '...(remaining elements truncated)...']>

In [31]:
periode_krs = 1
periode_krs = PeriodeKRS.objects.get(id=periode_krs)
        
mata_kuliah_qs = (
    MataKuliah.objects
    .filter(
        kelas__isnull=False,
        kelas__penawarankrs__periode_krs=periode_krs
    )
    .prefetch_related(
        Prefetch(
            "kelas_set",
            # queryset=Kelas.objects.filter(id=1),
            to_attr="daftar_kelas"
        )
    ).distinct()
)
mata_kuliah_qs
mata_kuliah_serializer = MataKuliahAjukanBanyakSerializer(mata_kuliah_qs, many=True)
mata_kuliah_serializer.data

[{'id': 61, 'nama': 'ADMINISTRASI SERVER (ROUTING & SWITCHING) + (PRAKT)', 'sks': 3, 'kode': '31MKP63AS19', 'semester': 5, 'jenis': None, 'daftar_kelas': [{'id': 125, 'nama': 'PPW', 'kode': '31MKP63AS19-PPW', 'waktu': None, 'ruangan': None}]}, {'id': 44, 'nama': 'AGAMA', 'sks': 2, 'kode': '11MKI00AG01', 'semester': 1, 'jenis': None, 'daftar_kelas': [{'id': 126, 'nama': 'PPW', 'kode': '11MKI63PA01-PPW', 'waktu': None, 'ruangan': None}]}, {'id': 41, 'nama': 'AKHLAK DAN BUDAYA BUTON', 'sks': 3, 'kode': '11MKI00AB06', 'semester': 1, 'jenis': None, 'daftar_kelas': [{'id': 127, 'nama': 'PPW', 'kode': '11MKI63AB04-PPW', 'waktu': None, 'ruangan': None}]}, {'id': 47, 'nama': 'APLIKASI KOMPUTER KANTOR + (PRAKTIKUM)', 'sks': 3, 'kode': '11MKP63AK06', 'semester': 1, 'jenis': None, 'daftar_kelas': [{'id': 128, 'nama': 'PPW', 'kode': '11MKP63AK06-PPW', 'waktu': None, 'ruangan': None}]}, {'id': 52, 'nama': 'ARSITEKTUR DAN ORGANISASI KOMPUTER', 'sks': 2, 'kode': '21MKP63AO16', 'semester': 3, 'jenis': 

In [52]:
mahasiswa_qs = (
    Mahasiswa.objects
    .filter(
        krs__detail__kelas__id=126
    )
    .distinct()
)

mahasiswa_qs
mahasiswa_serializer = MahasiswaAjukanBanyakSerializerSerializer(mahasiswa_qs, many=True)
mahasiswa_serializer.data

[{'id': 67, 'nama': 'ADITA AYU HAZRIN', 'nim': '23650238', 'no_hp': '082151029053', 'jenis_kelamin': None}, {'id': 161, 'nama': 'SANTI', 'nim': '25650264', 'no_hp': '082349142586', 'jenis_kelamin': None}]

In [86]:
angkatan = 2025
prefix = str(angkatan)[-2:]

mahasiswa_qs = (
    Mahasiswa.objects
    .filter(
        nim__startswith=prefix,
        status__iexact="aktif"
    )
)

mahasiswa_qs
mahasiswa_serializer = MahasiswaAjukanBanyakSerializerSerializer(mahasiswa_qs, many=True)
mahasiswa_serializer.data

[{'id': 115, 'nama': 'WD. MARSYA. M', 'nim': '25650106', 'no_hp': '085294474749', 'jenis_kelamin': None}, {'id': 116, 'nama': 'NUR DIASI MANJALE', 'nim': '25650107', 'no_hp': '082113618081', 'jenis_kelamin': None}, {'id': 117, 'nama': 'HADIJA NURSALIMA', 'nim': '25650111', 'no_hp': '082338695506', 'jenis_kelamin': None}, {'id': 118, 'nama': 'SANIA', 'nim': '25650112', 'no_hp': '082231856236', 'jenis_kelamin': None}, {'id': 119, 'nama': 'AFRIZAL BANASAEMA ', 'nim': '25650113', 'no_hp': '082227104045', 'jenis_kelamin': None}, {'id': 120, 'nama': 'NUR AFNI OKTAVIA ', 'nim': '25650114', 'no_hp': '085141372185', 'jenis_kelamin': None}, {'id': 121, 'nama': 'SALMA', 'nim': '25650115', 'no_hp': '082213369850', 'jenis_kelamin': None}, {'id': 122, 'nama': 'WA ODE PUTRI MENTARI', 'nim': '25650116', 'no_hp': '085925351377', 'jenis_kelamin': None}, {'id': 123, 'nama': 'SYAQILA NUR AULIA', 'nim': '25650118', 'no_hp': '082246496012', 'jenis_kelamin': None}, {'id': 124, 'nama': 'SELFINA NADIL ', 'nim'

In [77]:
angkatan = 2025
prefix = str(angkatan)[-2:]

mahasiswa_qs = (
    Mahasiswa.objects
    .filter(
        nim__startswith=prefix,
        status__isnull=True
    )
)

mahasiswa_qs
mahasiswa_serializer = MahasiswaAjukanBanyakSerializerSerializer(mahasiswa_qs, many=True)
mahasiswa_serializer.data

[{'id': 115, 'nama': 'WD. MARSYA. M', 'nim': '25650106', 'no_hp': '085294474749', 'jenis_kelamin': None}, {'id': 116, 'nama': 'NUR DIASI MANJALE', 'nim': '25650107', 'no_hp': '082113618081', 'jenis_kelamin': None}, {'id': 117, 'nama': 'HADIJA NURSALIMA', 'nim': '25650111', 'no_hp': '082338695506', 'jenis_kelamin': None}, {'id': 118, 'nama': 'SANIA', 'nim': '25650112', 'no_hp': '082231856236', 'jenis_kelamin': None}, {'id': 119, 'nama': 'AFRIZAL BANASAEMA ', 'nim': '25650113', 'no_hp': '082227104045', 'jenis_kelamin': None}, {'id': 120, 'nama': 'NUR AFNI OKTAVIA ', 'nim': '25650114', 'no_hp': '085141372185', 'jenis_kelamin': None}, {'id': 121, 'nama': 'SALMA', 'nim': '25650115', 'no_hp': '082213369850', 'jenis_kelamin': None}, {'id': 122, 'nama': 'WA ODE PUTRI MENTARI', 'nim': '25650116', 'no_hp': '085925351377', 'jenis_kelamin': None}, {'id': 123, 'nama': 'SYAQILA NUR AULIA', 'nim': '25650118', 'no_hp': '082246496012', 'jenis_kelamin': None}, {'id': 124, 'nama': 'SELFINA NADIL ', 'nim'

query set & delete

In [ ]:
# tambah anggota kelas
kelas_id = 126
krs_ids = [1, 2, 3, 4]

detail_krs_list = []

# DetailKRS.objects.bulk_create(
#     detail_krs_list,
#     ignore_conflicts=True
# )

In [88]:
mahasiswa_qs = Mahasiswa.objects.all()

In [90]:
mahasiswa_qs.update(status="Aktif")
# mahasiswa_qs.update(prodi_id=1)

166

In [85]:
Mahasiswa.objects.get(id=117).prodi_id

1

In [ ]:
svc = AjukanBanyakService()
svc.daftar_mata_kuliah(periode_krs_id=1)